# 02 - Preprocessing do Dataset com Máscaras

Este notebook prepara todas as imagens do dataset aplicando um pipeline lesion-centric.

**Objetivos**

- validar imagens, máscaras e metadata
- remover artefatos de pelo
- fazer crop centrado na lesão (guiado pela máscara)
- aplicar padding para quadrado (sem resize)
- salvar todas as imagens tratadas em `data/processed/treated/`

> O split train/val/test e o balanceamento de classes serão feitos no notebook de data augmentation.

In [52]:
import json
import os
import random
import shutil
import warnings
from collections import Counter
from pathlib import Path

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from PIL import Image

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")


def resolve_repo_root() -> Path:
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    for candidate in candidates:
        if (candidate / "data" / "metadata.csv").exists() and (candidate / "docs").exists():
            return candidate
    raise FileNotFoundError("Nao foi possivel localizar a raiz com data/metadata.csv.")


ROOT_DIR = resolve_repo_root()
DATA_DIR = ROOT_DIR / "data"
IMG_DIR = DATA_DIR / "images"
MASK_DIR = DATA_DIR / "masks"
META_PATH = DATA_DIR / "metadata.csv"
OUTPUT_DIR = ROOT_DIR / "notebooks" / "outputs"
FIG_DIR = OUTPUT_DIR / "figures"
PREP_DIR = OUTPUT_DIR / "preprocessing"
PROCESSED_DIR = DATA_DIR / "processed"
TREATED_DIR = PROCESSED_DIR / "treated"

for directory in [FIG_DIR, PREP_DIR, PROCESSED_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(PREP_DIR / ".mplconfig"))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CLASSES = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]
MASK_THRESHOLD = 127
BBOX_MARGIN_RATIO = 0.10
HAIR_KERNEL_SIZE = 17
HAIR_THRESHOLD = 10
HAIR_INPAINT_RADIUS = 1
EXPORT_IMAGE_FORMAT = "png"

config = {
    "seed": SEED,
    "mask_threshold": MASK_THRESHOLD,
    "bbox_margin_ratio": BBOX_MARGIN_RATIO,
    "hair_kernel_size": HAIR_KERNEL_SIZE,
    "hair_threshold": HAIR_THRESHOLD,
    "hair_inpaint_radius": HAIR_INPAINT_RADIUS,
    "export_image_format": EXPORT_IMAGE_FORMAT,
    "treated_dir": str(TREATED_DIR),
}

print(json.dumps(config, indent=2))

{
  "seed": 42,
  "mask_threshold": 127,
  "bbox_margin_ratio": 0.1,
  "hair_kernel_size": 17,
  "hair_threshold": 10,
  "hair_inpaint_radius": 1,
  "export_image_format": "png",
  "treated_dir": "/Users/eduardoyaginuma/Documents/Repositorios/insper/skin-cancer-images-segmentation/data/processed/treated"
}


## 1. Carregamento do dataset bruto
        

In [53]:
raw_df = pd.read_csv(META_PATH).copy()

assert set(CLASSES).issubset(raw_df.columns), "Metadata sem uma ou mais classes."
assert (raw_df[CLASSES].sum(axis=1) == 1).all(), "Cada imagem deve pertencer a uma unica classe."

raw_df["label"] = raw_df[CLASSES].idxmax(axis=1)
raw_df["binary_label"] = (raw_df["label"] == "MEL").astype(int)
raw_df["img_path"] = raw_df["image"].map(lambda image_id: IMG_DIR / f"{image_id}.jpg")
raw_df["mask_path"] = raw_df["image"].map(lambda image_id: MASK_DIR / f"{image_id}.png")

raw_summary = pd.DataFrame(
    [
        {"metrica": "linhas_metadata", "valor": len(raw_df)},
        {"metrica": "imagens_faltantes", "valor": int((~raw_df["img_path"].map(Path.exists)).sum())},
        {"metrica": "masks_faltantes", "valor": int((~raw_df["mask_path"].map(Path.exists)).sum())},
        {"metrica": "melanomas", "valor": int((raw_df["binary_label"] == 1).sum())},
        {"metrica": "nao_melanomas", "valor": int((raw_df["binary_label"] == 0).sum())},
    ]
)

label_summary = raw_df["label"].value_counts().reindex(CLASSES).rename_axis("classe").reset_index(name="count")
label_summary["pct"] = (label_summary["count"] / len(raw_df) * 100).round(2)

display(raw_summary)
display(label_summary)
print(
    f"Resumo rapido: {len(raw_df):,} imagens, "
    f"{int((raw_df['binary_label'] == 1).sum()):,} melanomas e "
    f"{int((raw_df['binary_label'] == 0).sum()):,} nao melanomas."
)

# Todas as imagens sao incluidas no preprocessing (sem downsampling)
df = raw_df.copy()

,metrica,valor
0,linhas_metadata,10015
1,imagens_faltantes,0
2,masks_faltantes,0
3,melanomas,1113
4,nao_melanomas,8902


,classe,count,pct
0,MEL,1113,11.11
1,NV,6705,66.95
2,BCC,514,5.13
3,AKIEC,327,3.27
4,BKL,1099,10.97
5,DF,115,1.15
6,VASC,142,1.42


Resumo rapido: 10,015 imagens, 1,113 melanomas e 8,902 nao melanomas.


## 2. Checagens de integridade

Antes do preprocessing, confirmamos que imagens e máscaras batem com a metadata e que os shapes são consistentes.

In [54]:
def load_mask_array(mask_path: Path) -> np.ndarray:
    return (np.array(Image.open(mask_path).convert("L")) > MASK_THRESHOLD).astype(np.uint8)


image_sizes = Counter()
mask_sizes = Counter()
mask_coverages = []
bbox_height_ratios = []
bbox_width_ratios = []

missing_images = []
missing_masks = []

for _, row in df.iterrows():
    img_path = row["img_path"]
    mask_path = row["mask_path"]

    if not img_path.exists():
        missing_images.append(str(img_path))
        continue
    if not mask_path.exists():
        missing_masks.append(str(mask_path))
        continue

    with Image.open(img_path) as image_file:
        image_sizes[image_file.size] += 1

    mask = load_mask_array(mask_path)
    mask_sizes[(mask.shape[1], mask.shape[0])] += 1
    mask_coverages.append(mask.mean())

    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        bbox_width_ratios.append(0.0)
        bbox_height_ratios.append(0.0)
    else:
        bbox_width_ratios.append((xs.max() - xs.min() + 1) / mask.shape[1])
        bbox_height_ratios.append((ys.max() - ys.min() + 1) / mask.shape[0])

df["mask_coverage"] = mask_coverages
df["bbox_width_ratio"] = bbox_width_ratios
df["bbox_height_ratio"] = bbox_height_ratios

integrity_summary = pd.DataFrame(
    {
        "metrica": [
            "total_imagens",
            "imagens_unicas",
            "imagens_faltantes",
            "masks_faltantes",
            "distinct_image_sizes",
            "distinct_mask_sizes",
        ],
        "valor": [
            len(df),
            df["image"].nunique(),
            len(missing_images),
            len(missing_masks),
            dict(image_sizes),
            dict(mask_sizes),
        ],
    }
)

display(integrity_summary)

assert len(missing_images) == 0, "Existem imagens faltantes."
assert len(missing_masks) == 0, "Existem mascaras faltantes."
assert df["image"].nunique() == len(df), "Existem image ids duplicados."
assert len(image_sizes) == 1, "As imagens nao possuem tamanho unico."
assert len(mask_sizes) == 1, "As mascaras nao possuem tamanho unico."

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["mask_coverage"], bins=40, kde=True, ax=axes[0], color="#2b6cb0")
axes[0].set_title("Distribuicao da Cobertura das Mascaras")
axes[0].set_xlabel("Cobertura da mascara")

sns.scatterplot(
    data=df.sample(min(1500, len(df)), random_state=SEED),
    x="bbox_width_ratio",
    y="bbox_height_ratio",
    hue="binary_label",
    palette={0: "#2b6cb0", 1: "#c53030"},
    alpha=0.7,
    ax=axes[1],
)
axes[1].set_title("Bounding Box por Amostra")
axes[1].set_xlabel("bbox width / image width")
axes[1].set_ylabel("bbox height / image height")

plt.tight_layout()
plt.savefig(FIG_DIR / "preprocessing_integrity.png", dpi=150, bbox_inches="tight")
plt.show()

print("Checagens de integridade concluidas: imagens e mascaras estao consistentes.")

,metrica,valor
0,total_imagens,10015
1,imagens_unicas,10015
2,imagens_faltantes,0
3,masks_faltantes,0
4,distinct_image_sizes,"{(600, 450): 10015}"
5,distinct_mask_sizes,"{(600, 450): 10015}"


Checagens de integridade concluidas: imagens e mascaras estao consistentes.


## 3. Preprocessing lesion-centric

O pipeline determinístico usa a máscara para localizar a lesão:

1. carregar imagem RGB e máscara binária
2. remover pelos finos
3. localizar a bounding box da lesão
4. expandir a região com uma margem de segurança (10%)
5. fazer crop lesion-centric
6. aplicar padding para quadrado (tamanho natural preservado, sem resize)

In [55]:
def load_rgb_image(image_path: Path) -> np.ndarray:
    return np.array(Image.open(image_path).convert("RGB"))


def remove_hair_artifacts(
    image_rgb: np.ndarray,
    kernel_size: int = HAIR_KERNEL_SIZE,
    threshold: int = HAIR_THRESHOLD,
    inpaint_radius: int = HAIR_INPAINT_RADIUS,
) -> tuple[np.ndarray, np.ndarray]:
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, hair_mask = cv2.threshold(blackhat, threshold, 255, cv2.THRESH_BINARY)
    cleaned = cv2.inpaint(image_rgb, hair_mask, inpaint_radius, cv2.INPAINT_TELEA)
    return cleaned, hair_mask


def compute_mask_bbox(mask: np.ndarray) -> tuple[int, int, int, int]:
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        height, width = mask.shape
        return 0, 0, width, height
    return xs.min(), ys.min(), xs.max() + 1, ys.max() + 1


def expand_bbox(
    bbox: tuple[int, int, int, int],
    width: int,
    height: int,
    margin_ratio: float = BBOX_MARGIN_RATIO,
) -> tuple[int, int, int, int]:
    x0, y0, x1, y1 = bbox
    box_width = x1 - x0
    box_height = y1 - y0
    margin = max(int(max(box_width, box_height) * margin_ratio), 1)

    x0 = max(0, x0 - margin)
    y0 = max(0, y0 - margin)
    x1 = min(width, x1 + margin)
    y1 = min(height, y1 + margin)
    return x0, y0, x1, y1


def crop_array(array: np.ndarray, bbox: tuple[int, int, int, int]) -> np.ndarray:
    x0, y0, x1, y1 = bbox
    return array[y0:y1, x0:x1]


def pad_to_square(array: np.ndarray) -> np.ndarray:
    height, width = array.shape[:2]
    side = max(height, width)

    pad_top = (side - height) // 2
    pad_bottom = side - height - pad_top
    pad_left = (side - width) // 2
    pad_right = side - width - pad_left

    if array.ndim == 3:
        pad_width = ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0))
    else:
        pad_width = ((pad_top, pad_bottom), (pad_left, pad_right))

    return np.pad(array, pad_width=pad_width, mode="edge")


def preprocess_image_and_mask(
    image_path: Path,
    mask_path: Path,
    remove_hair: bool = True,
) -> tuple[np.ndarray, np.ndarray, dict]:
    image = load_rgb_image(image_path)
    mask = load_mask_array(mask_path)

    if remove_hair:
        image, hair_mask = remove_hair_artifacts(image)
    else:
        hair_mask = np.zeros(mask.shape, dtype=np.uint8)

    bbox = compute_mask_bbox(mask)
    bbox = expand_bbox(bbox, width=image.shape[1], height=image.shape[0])

    image = crop_array(image, bbox)
    mask = crop_array(mask, bbox)

    image = pad_to_square(image)
    mask = pad_to_square(mask)
    mask = (mask > 0).astype(np.uint8)

    debug = {
        "bbox": bbox,
        "final_shape": image.shape[:2],
        "mask_coverage_after_crop": float(mask.mean()),
        "hair_pixels_detected": int((hair_mask > 0).sum()),
    }
    return image, mask, debug

In [56]:
preview_rows = []
for label in ["MEL", "NV", "BKL"]:
    preview_rows.append(df[df["label"] == label].sample(1, random_state=SEED).iloc[0])

fig, axes = plt.subplots(len(preview_rows), 4, figsize=(16, 4 * len(preview_rows)))

for row_idx, row in enumerate(preview_rows):
    raw_image = load_rgb_image(row["img_path"])
    raw_mask = load_mask_array(row["mask_path"])
    hair_removed, hair_mask = remove_hair_artifacts(raw_image)
    processed_image, processed_mask, debug = preprocess_image_and_mask(
        row["img_path"],
        row["mask_path"],
        remove_hair=True,
    )

    axes[row_idx, 0].imshow(raw_image)
    axes[row_idx, 0].set_title(f"{row['label']} - original")
    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(hair_mask, cmap="gray")
    axes[row_idx, 1].set_title("mascara de pelos")
    axes[row_idx, 1].axis("off")

    axes[row_idx, 2].imshow(hair_removed)
    axes[row_idx, 2].contour(raw_mask, levels=[0.5], colors="lime", linewidths=1)
    axes[row_idx, 2].set_title("hair removed + contorno")
    axes[row_idx, 2].axis("off")

    axes[row_idx, 3].imshow(processed_image)
    axes[row_idx, 3].set_title(f"processada {debug['final_shape'][0]}x{debug['final_shape'][1]}")
    axes[row_idx, 3].axis("off")

plt.tight_layout()
plt.savefig(FIG_DIR / "preprocessing_steps.png", dpi=150, bbox_inches="tight")
plt.show()

# Verificar que todas as imagens processadas sao quadradas
uniformity_checks = []
for _, row in df.sample(min(64, len(df)), random_state=SEED).iterrows():
    processed_image, processed_mask, _ = preprocess_image_and_mask(
        row["img_path"],
        row["mask_path"],
        remove_hair=True,
    )
    h, w = processed_image.shape[:2]
    uniformity_checks.append(h == w)

assert all(uniformity_checks), "Existem imagens nao quadradas apos preprocessing."
print(f"Todas as {len(uniformity_checks)} amostras testadas sao quadradas apos preprocessing.")

Todas as 64 amostras testadas sao quadradas apos preprocessing.


## 4. Exportação das imagens tratadas

Todas as imagens são processadas e salvas em uma única pasta `data/processed/treated/`.

O split train/val/test e o balanceamento de classes serão feitos no notebook de data augmentation.

In [57]:
def save_rgb_image(image_array: np.ndarray, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(image_array.astype(np.uint8)).save(output_path)


if TREATED_DIR.exists():
    shutil.rmtree(TREATED_DIR)
TREATED_DIR.mkdir(parents=True, exist_ok=True)

manifest_rows = []
total = len(df)

for i, (_, row) in enumerate(df.iterrows(), start=1):
    image, mask, debug = preprocess_image_and_mask(
        row["img_path"],
        row["mask_path"],
        remove_hair=True,
    )
    output_path = TREATED_DIR / f"{row['image']}.{EXPORT_IMAGE_FORMAT}"
    save_rgb_image(image, output_path)
    manifest_rows.append(
        {
            "image_id": row["image"],
            "label": row["label"],
            "binary_label": row["binary_label"],
            "export_path": str(output_path),
            "final_height": debug["final_shape"][0],
            "final_width": debug["final_shape"][1],
            "mask_coverage_after_crop": debug["mask_coverage_after_crop"],
            "hair_pixels_detected": debug["hair_pixels_detected"],
        }
    )
    if i % 500 == 0 or i == total:
        print(f"  {i}/{total} imagens processadas...")

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(PROCESSED_DIR / "treated_manifest.csv", index=False)

export_summary = pd.DataFrame(
    [
        {"metrica": "total_imagens_exportadas", "valor": len(manifest_df)},
        {"metrica": "melanomas", "valor": int((manifest_df["binary_label"] == 1).sum())},
        {"metrica": "nao_melanomas", "valor": int((manifest_df["binary_label"] == 0).sum())},
        {"metrica": "pasta", "valor": str(TREATED_DIR)},
    ]
)
display(export_summary)
print(f"\nExportacao concluida: {len(manifest_df):,} imagens salvas em {TREATED_DIR}")

  500/10015 imagens processadas...
  1000/10015 imagens processadas...
  1500/10015 imagens processadas...
  2000/10015 imagens processadas...
  2500/10015 imagens processadas...
  3000/10015 imagens processadas...
  3500/10015 imagens processadas...
  4000/10015 imagens processadas...
  4500/10015 imagens processadas...
  5000/10015 imagens processadas...
  5500/10015 imagens processadas...
  6000/10015 imagens processadas...
  6500/10015 imagens processadas...
  7000/10015 imagens processadas...
  7500/10015 imagens processadas...
  8000/10015 imagens processadas...
  8500/10015 imagens processadas...
  9000/10015 imagens processadas...
  9500/10015 imagens processadas...
  10000/10015 imagens processadas...
  10015/10015 imagens processadas...


,metrica,valor
0,total_imagens_exportadas,10015
1,melanomas,1113
2,nao_melanomas,8902
3,pasta,/Users/eduardoyaginuma/Documents/Repositorios/...



Exportacao concluida: 10,015 imagens salvas em /Users/eduardoyaginuma/Documents/Repositorios/insper/skin-cancer-images-segmentation/data/processed/treated


## 5. Conclusões

In [58]:
treated_count = len(list(TREATED_DIR.glob(f"*.{EXPORT_IMAGE_FORMAT}")))

summary_table = pd.DataFrame(
    [
        {"metrica": "total_imagens_no_dataset", "valor": len(raw_df)},
        {"metrica": "total_imagens_tratadas", "valor": treated_count},
        {"metrica": "melanomas", "valor": int((raw_df["binary_label"] == 1).sum())},
        {"metrica": "nao_melanomas", "valor": int((raw_df["binary_label"] == 0).sum())},
    ]
)
display(summary_table)

print("Conclusoes:")
print(f"- Todas as {len(raw_df):,} imagens foram tratadas sem resize.")
print( "- Pipeline aplicado: remocao de pelos -> bounding box -> crop lesion-centric -> padding para quadrado.")
print(f"- Imagens salvas em pasta unica: {TREATED_DIR}")
print( "- O split train/val/test e o balanceamento de classes serao feitos no notebook de data augmentation.")

,metrica,valor
0,total_imagens_no_dataset,10015
1,total_imagens_tratadas,10015
2,melanomas,1113
3,nao_melanomas,8902


Conclusoes:
- Todas as 10,015 imagens foram tratadas sem resize.
- Pipeline aplicado: remocao de pelos -> bounding box -> crop lesion-centric -> padding para quadrado.
- Imagens salvas em pasta unica: /Users/eduardoyaginuma/Documents/Repositorios/insper/skin-cancer-images-segmentation/data/processed/treated
- O split train/val/test e o balanceamento de classes serao feitos no notebook de data augmentation.
